In [ ]:
%pip install -q vllm sentence-transformers tqdm implicit

In [ ]:
import os

os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'

import gc
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../../data/children_products')
CLEANED_CSV = DATA_DIR / 'clildren_product_cleaned.csv'
RAW_CSV = DATA_DIR / '!03&04_17_VSE.csv'

MIN_INTERACTIONS = 5
TRAIN_SPLIT = 0.8

N_EVAL_USERS = 1000
K_CANDIDATES = 50
FINAL_K = 20
MAX_HISTORY_ITEMS = 20

BATCH_SIZE = 10 
SCORE_MIN, SCORE_MAX = 1, 5

MODEL_CHOICE = 'qwen7b'  # 'qwen3b' | 'qwen7b' | 'mistral7b' | 'vikhr12b' | 'qwen14b' | 'qwen32b_awq'
MODEL_REGISTRY = {
    'qwen3b':      ('Qwen/Qwen2.5-3B-Instruct',                          'qwen3b',      None,  6,  'qwen3b'),
    'qwen7b':      ('Qwen/Qwen2.5-7B-Instruct',                          'qwen7b',      None,  14, 'qwen7b'),
    'mistral7b':   ('mistralai/Mistral-7B-Instruct-v0.3',                'mistral7b',   None,  14, 'mistral7b'),
    'vikhr12b':    ('Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24',    'vikhr12b',    None,  24, 'vikhr12b'),
    'qwen14b':     ('Qwen/Qwen2.5-14B-Instruct',                         'qwen14b',     None,  28, 'qwen14b'),
    'qwen32b_awq': ('Qwen/Qwen2.5-32B-Instruct-AWQ',                     'qwen32b_awq', 'awq', 17, 'qwen32b_awq'),
}
LLM_MODEL, MODEL_SUFFIX, LLM_QUANT, _vram_est, _note = MODEL_REGISTRY[MODEL_CHOICE]
print(f'Using LLM: {LLM_MODEL}  (~{_vram_est} GB, {_note})')
LLM_RERANK_PW_CACHE = DATA_DIR / f'llm_rerank_pointwise_v2_cache_vllm_{MODEL_SUFFIX}.json'
CANDIDATES_CACHE = DATA_DIR / 'rerank_base_candidates.npz'
NCF_CHECKPOINT = DATA_DIR / 'ncf_state.pt'

RRF_K = 60
ALPHA_SWEEP = [0.0, 0.1, 0.3, 0.5, 1.0]

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


In [ ]:
df = pd.read_csv(CLEANED_CSV)
raw = pd.read_csv(
    RAW_CSV,
    sep=';', encoding='cp1251',
    usecols=['ID_SKU', 'Номенклатура', 'Группа2', 'Группа3', 'Тип'],
    dtype=str, low_memory=False
)
raw = raw.dropna(subset=['ID_SKU']).drop_duplicates('ID_SKU')

df = df.merge(raw[['ID_SKU', 'Номенклатура']], on='ID_SKU', how='left')
fallback = df['Группа3'].fillna('') + ' / ' + df['Тип'].fillna('')
df['Номенклатура'] = df['Номенклатура'].fillna(fallback)

df_filtered = df[(df['Статус'] == 'Доставлен') & (df['Отменено'] == 'Нет')].copy()
df_filtered = df_filtered.dropna(subset=['Телефон_new', 'ID_SKU', 'Дата'])
df_filtered['Дата'] = pd.to_datetime(df_filtered['Дата'], errors='coerce')
df_filtered = df_filtered.dropna(subset=['Дата'])

for _ in range(5):
    uc = df_filtered.groupby('Телефон_new').size()
    ic = df_filtered.groupby('ID_SKU').size()
    active_users = uc[uc >= MIN_INTERACTIONS].index
    active_items = ic[ic >= MIN_INTERACTIONS].index
    before = len(df_filtered)
    df_filtered = df_filtered[
        df_filtered['Телефон_new'].isin(active_users) &
        df_filtered['ID_SKU'].isin(active_items)
    ]
    if len(df_filtered) == before:
        break

user_enc, item_enc = LabelEncoder(), LabelEncoder()
df_filtered['user_id'] = user_enc.fit_transform(df_filtered['Телефон_new'])
df_filtered['item_id'] = item_enc.fit_transform(df_filtered['ID_SKU'])
n_users, n_items = len(user_enc.classes_), len(item_enc.classes_)
print(f'Пользователей: {n_users:,}  Товаров: {n_items:,}')

df_filtered = df_filtered.sort_values('Дата')
split_date = df_filtered['Дата'].quantile(TRAIN_SPLIT)
train_data = df_filtered[df_filtered['Дата'] < split_date].copy()
test_data = df_filtered[df_filtered['Дата'] >= split_date].copy()

train_int = train_data.groupby(['user_id', 'item_id']).size().reset_index(name='count')
test_int = test_data.groupby(['user_id', 'item_id']).size().reset_index(name='count')
train_matrix = csr_matrix(
    (train_int['count'].values, (train_int['user_id'].values, train_int['item_id'].values)),
    shape=(n_users, n_items)
)

test_user_items = test_int.groupby('user_id')['item_id'].apply(list).to_dict()
train_user_set = set(train_int['user_id'].unique())
warm_eval_users = [u for u in test_user_items if u in train_user_set]

rng = np.random.default_rng(RANDOM_SEED)
sample_users = rng.choice(warm_eval_users, size=min(N_EVAL_USERS, len(warm_eval_users)), replace=False)
sample_users = sorted(map(int, sample_users))
print(f'Sample users: {len(sample_users)}')


In [ ]:
# User features
user_features = train_data.groupby('user_id').agg(
    user_n_purchases=('user_id', 'size'),
    user_n_unique_items=('item_id', 'nunique'),
    user_avg_price=('Цена', 'mean'),
).reset_index()

user_geo = train_data.groupby('user_id')['Гео'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) else 'Неизвестно'
).reset_index()
user_geo.columns = ['user_id', 'user_geo']

user_delivery = train_data.groupby('user_id')['МетодДоставки_Групп'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) else 'Неизвестно'
).reset_index()
user_delivery.columns = ['user_id', 'user_delivery_method']

user_features = user_features.merge(user_geo, on='user_id').merge(user_delivery, on='user_id')

user_top_cats = (
    train_data.groupby(['user_id', 'Группа2']).size()
    .reset_index(name='cnt')
    .sort_values(['user_id', 'cnt'], ascending=[True, False])
    .groupby('user_id')['Группа2']
    .apply(lambda s: list(s.head(3)))
    .reset_index()
)
user_top_cats.columns = ['user_id', 'user_top_cats']
user_features = user_features.merge(user_top_cats, on='user_id', how='left')
user_features['user_top_cats'] = user_features['user_top_cats'].apply(
    lambda x: x if isinstance(x, list) else []
)
user_feat_dict = user_features.set_index('user_id').to_dict('index')

# Item features
item_features = train_data.groupby('item_id').agg(
    item_avg_price=('Цена', 'mean'),
    item_n_purchases=('item_id', 'size'),
).reset_index()
for col in ['Группа2', 'Группа3', 'Тип']:
    if col in train_data.columns:
        modal = train_data.groupby('item_id')[col].agg(
            lambda x: x.mode().iloc[0] if len(x.mode()) else 'Неизвестно'
        ).reset_index()
        modal.columns = ['item_id', col]
        item_features = item_features.merge(modal, on='item_id', how='left')
item_feat_dict = item_features.set_index('item_id').to_dict('index')

item_name_map = (
    df_filtered.groupby('item_id')['Номенклатура']
    .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
)
print(f'User feats: {len(user_feat_dict)}, item feats: {len(item_feat_dict)}')


In [ ]:
train_sorted = train_data.sort_values('Дата')
user_history_cache = {}
for uid, g in train_sorted.groupby('user_id'):
    user_history_cache[int(uid)] = list(zip(g['Дата'].dt.date.astype(str), g['Номенклатура']))
user_train_items_cache = {
    int(uid): set(g['item_id'].tolist())
    for uid, g in train_sorted.groupby('user_id')
}


def build_user_history(user_id: int, max_items: int = MAX_HISTORY_ITEMS) -> str:
    items = user_history_cache.get(user_id, [])
    tail = items[-max_items:]
    return '\n'.join(f'- {date}: {name}' for date, name in tail)


In [ ]:
def load_candidates_or_none(path: Path):
    if not path.exists():
        return None
    data = np.load(path, allow_pickle=False)
    users = [int(u) for u in data['users']]
    return {
        'users': users,
        'als': {users[i]: list(zip(data['als_ids'][i].tolist(),
                                     data['als_scores'][i].astype(float).tolist()))
                for i in range(len(users))},
        'ncf': {users[i]: list(zip(data['ncf_ids'][i].tolist(),
                                     data['ncf_scores'][i].astype(float).tolist()))
                for i in range(len(users))},
    }


loaded = load_candidates_or_none(CANDIDATES_CACHE)
if loaded is not None and set(loaded['users']) >= set(sample_users):
    als_candidates = {u: loaded['als'][u] for u in sample_users}
    ncf_candidates = {u: loaded['ncf'][u] for u in sample_users}
    print(f'Кандидаты восстановлены из {CANDIDATES_CACHE}')
    skip_training = True
else:
    print(f'Кэш кандидатов не найден ({CANDIDATES_CACHE.exists()=}, '
          f'нужно дообучить базы для {len(set(sample_users) - set(loaded["users"]) if loaded else sample_users)} юзеров)')
    skip_training = False


In [ ]:
if not skip_training:
    try:
        from implicit.gpu.als import AlternatingLeastSquares
    except ImportError:
        from implicit.als import AlternatingLeastSquares

    als = AlternatingLeastSquares(
        factors=34, regularization=0.2426, iterations=15,
        calculate_training_loss=True, random_state=RANDOM_SEED
    )
    als.fit(train_matrix)

    als_candidates = {}
    for uid in tqdm(sample_users, desc='ALS top-50'):
        ids, scores = als.recommend(
            uid, train_matrix[uid],
            N=K_CANDIDATES, filter_already_liked_items=True
        )
        als_candidates[int(uid)] = list(zip([int(x) for x in ids], [float(s) for s in scores]))

    if NCF_CHECKPOINT.exists():
        class NCF(nn.Module):
            def __init__(self, num_users, num_items, emb_dim=62, dropout=0.4935050):
                super().__init__()
                self.user_embedding_gmf = nn.Embedding(num_users, emb_dim)
                self.item_embedding_gmf = nn.Embedding(num_items, emb_dim)
                self.user_embedding_mlp = nn.Embedding(num_users, emb_dim)
                self.item_embedding_mlp = nn.Embedding(num_items, emb_dim)
                self.mlp = nn.Sequential(
                    nn.Linear(emb_dim * 2, 128),
                    nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
                    nn.Linear(128, 64),
                    nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
                )
                self.fc_out = nn.Linear(emb_dim + 64, 1)

            def forward(self, user_ids, item_ids):
                u_gmf = self.user_embedding_gmf(user_ids)
                v_gmf = self.item_embedding_gmf(item_ids)
                gmf_out = u_gmf * v_gmf
                u_mlp = self.user_embedding_mlp(user_ids)
                v_mlp = self.item_embedding_mlp(item_ids)
                mlp_out = self.mlp(torch.cat([u_mlp, v_mlp], dim=-1))
                return torch.sigmoid(self.fc_out(torch.cat([gmf_out, mlp_out], dim=-1)).squeeze(-1))

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        ncf_model = NCF(n_users, n_items, emb_dim=62, dropout=0.4935050).to(device)
        ncf_model.load_state_dict(torch.load(NCF_CHECKPOINT, map_location=device))
        ncf_model.eval()

        ncf_candidates = {}
        items_t = torch.arange(n_items, device=device, dtype=torch.long)
        with torch.no_grad():
            for uid in tqdm(sample_users, desc='NCF top-50'):
                users_t = torch.full_like(items_t, fill_value=int(uid))
                scores = ncf_model(users_t, items_t).cpu().numpy()
                scores = np.nan_to_num(scores, nan=-np.inf)
                bought = user_train_items_cache.get(int(uid), set())
                if bought:
                    scores[list(bought)] = -np.inf
                top_idx = np.argpartition(-scores, K_CANDIDATES)[:K_CANDIDATES]
                top_idx = top_idx[np.argsort(-scores[top_idx])]
                ncf_candidates[int(uid)] = [(int(i), float(scores[i])) for i in top_idx]
        del ncf_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        print('NCF checkpoint не найден — оцениваем только ALS')
        ncf_candidates = {u: [] for u in sample_users}



In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

_eot_id = tokenizer.convert_tokens_to_ids('<|eot_id|>')
LLM_TERMINATORS = sorted({
    int(t) for t in [tokenizer.eos_token_id, _eot_id]
    if isinstance(t, int) and t is not None and t >= 0
})

_llm_kwargs = dict(
    model=LLM_MODEL,
    max_model_len=8192,
    gpu_memory_utilization=0.9,
    enforce_eager=False,
    trust_remote_code=False,
)
if LLM_QUANT is None:
    _llm_kwargs['dtype'] = 'bfloat16'
else:
    _llm_kwargs['quantization'] = LLM_QUANT
    _llm_kwargs['dtype'] = 'auto'
llm = LLM(**_llm_kwargs)
print('vLLM loaded')


In [ ]:
SYSTEM_PROMPT_PW = (
    'Ты — ассистент-рекомендатель детского интернет-магазина. Тебе дан профиль пользователя, '
    'его история покупок и небольшой батч из {n} товаров. '
    'Твоя задача — оценить каждый товар по шкале от {smin} до {smax}, насколько вероятно, что '
    'этот пользователь купит его в ближайшее время. '
    'Шкала: 1 = очень маловероятно, 3 = средне, 5 = очень вероятно. '
    'Опирайся на возраст ребёнка (выводимый из истории), бренды, категории, ценовой сегмент. '
    'Используй РАЗНЫЕ значения — не ставь всем одинаковую оценку. '
    'Отвечай СТРОГО в формате JSON без пояснений: '
    '{{"scores": [{{"id": 1, "score": <число>}}, ..., {{"id": {n}, "score": <число>}}]}}.'
)


def format_user_profile(uid: int) -> str:
    feat = user_feat_dict.get(int(uid), {})
    if not feat:
        return 'Профиль пользователя: нет данных.'
    top_cats = feat.get('user_top_cats') or []
    top_cats_str = ', '.join(top_cats) if top_cats else 'нет данных'
    avg_price = feat.get('user_avg_price', 0) or 0
    return (
        'Профиль пользователя:\n'
        f'- Топ-категории: {top_cats_str}\n'
        f'- Средний чек: {avg_price:.0f} руб\n'
        f'- География: {feat.get("user_geo", "неизвестно")}\n'
        f'- Кол-во заказов: {int(feat.get("user_n_purchases", 0))}\n'
        f'- Доставка: {feat.get("user_delivery_method", "неизвестно")}'
    )


def format_candidate_line(idx: int, iid: int) -> str:
    name = item_name_map.loc[iid]
    feat = item_feat_dict.get(int(iid), {})
    cat2 = feat.get('Группа2', 'н/д')
    cat3 = feat.get('Группа3', 'н/д')
    typ = feat.get('Тип', 'н/д')
    price = feat.get('item_avg_price', 0) or 0
    return (
        f'{idx + 1}. {name} | {cat2} > {cat3} | тип: {typ} | цена: {price:.0f}р'
    )


def build_pointwise_prompt(user_id: int, candidate_item_ids):
    profile = format_user_profile(user_id)
    hist = build_user_history(user_id, max_items=MAX_HISTORY_ITEMS)
    cand_lines = '\n'.join(format_candidate_line(idx, iid) for idx, iid in enumerate(candidate_item_ids))
    n = len(candidate_item_ids)
    user_prompt = (
        f'{profile}\n\n'
        f'История покупок (последние {MAX_HISTORY_ITEMS}):\n{hist}\n\n'
        f'Оцени каждый из {n} товаров по шкале от {SCORE_MIN} до {SCORE_MAX}:\n{cand_lines}\n\n'
        f'Верни JSON: {{"scores": [{{"id": 1, "score": <число>}}, ..., {{"id": {n}, "score": <число>}}]}}. '
        f'Только JSON, без поясняющего текста.'
    )
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT_PW.format(n=n, smin=SCORE_MIN, smax=SCORE_MAX)},
        {'role': 'user', 'content': user_prompt},
    ]
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)


# Демо
demo_uid = sample_users[0]
demo_cands = [iid for iid, _ in als_candidates[demo_uid][:BATCH_SIZE]]
print(build_pointwise_prompt(demo_uid, demo_cands)[:1500])
print('...')


In [ ]:
def parse_scores_json(raw_text: str, n_in_batch: int):
    """Возвращает np.array(n_in_batch,) с скорами или None при невосстановимой ошибке.

    Пустые позиции заполняются средним по батчу. id ожидается 1-индексированный."""
    arr = None
    try:
        obj = json.loads(raw_text)
        if isinstance(obj, dict):
            for key in ('scores', 'ratings', 'items'):
                if key in obj and isinstance(obj[key], list):
                    arr = obj[key]
                    break
            if arr is None:
                for v in obj.values():
                    if isinstance(v, list):
                        arr = v
                        break
        elif isinstance(obj, list):
            arr = obj
    except Exception:
        m = re.search(r'\[[\s\S]*?\]', raw_text)
        if m:
            try:
                arr = json.loads(m.group(0))
            except Exception:
                arr = None
    if arr is None:
        return None

    scores = np.full(n_in_batch, np.nan, dtype=float)
    for entry in arr:
        if not isinstance(entry, dict):
            continue
        try:
            i = int(entry.get('id', -1)) - 1
            s = float(entry.get('score', np.nan))
        except Exception:
            continue
        if 0 <= i < n_in_batch and not np.isnan(s):
            s = max(SCORE_MIN, min(SCORE_MAX, s))
            scores[i] = s

    if np.all(np.isnan(scores)):
        return None
    # Заполняем пропуски средним по батчу
    mean = float(np.nanmean(scores))
    scores = np.where(np.isnan(scores), mean, scores)
    return scores


In [ ]:
pw_cache = {
    'als_scores': {}, 'ncf_scores': {},
    'als_raw': {}, 'ncf_raw': {},
}
if LLM_RERANK_PW_CACHE.exists():
    with open(LLM_RERANK_PW_CACHE, 'r', encoding='utf-8') as f:
        loaded = json.load(f)
    for k in pw_cache:
        if k.endswith('_scores'):
            pw_cache[k] = {int(uid): np.asarray(v, dtype=float)
                           for uid, v in loaded.get(k, {}).items()}
        else:
            pw_cache[k] = {int(uid): v for uid, v in loaded.get(k, {}).items()}
    print(f'Cache: восстановлено {sum(len(v) for k, v in pw_cache.items() if k.endswith("_scores"))} рядов скоров')

sources = [
    ('als', als_candidates, 'als_scores', 'als_raw'),
    ('ncf', ncf_candidates, 'ncf_scores', 'ncf_raw'),
]


def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield i, seq[i:i + size]

tasks = []
prompts = []
for src_name, cand_dict, score_key, raw_key in sources:
    for uid in sample_users:
        uid = int(uid)
        if not cand_dict[uid]:
            continue
        if uid in pw_cache[score_key]:
            continue
        cand_ids = [iid for iid, _ in cand_dict[uid]]
        for offset, batch in chunked(cand_ids, BATCH_SIZE):
            tasks.append((score_key, raw_key, uid, offset, len(batch)))
            prompts.append(build_pointwise_prompt(uid, batch))

print(f'Промптов к обработке: {len(prompts)}')

if prompts:
    sample_lens = [len(tokenizer.encode(p)) for p in prompts[:50]]
    p50, p95, p99 = np.percentile(sample_lens, [50, 95, 99])
    print(f'Длина промпта: p50={p50:.0f}, p95={p95:.0f}, p99={p99:.0f}')

    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=300,
        stop_token_ids=LLM_TERMINATORS,
    )

    t0 = time.time()
    outputs = llm.generate(prompts, sampling_params)
    elapsed = time.time() - t0
    print(f'\nGenerated {len(prompts)} prompts in {elapsed:.1f}s ({elapsed/len(prompts):.2f}s/prompt)')

    pending = {}
    raw_pending = {}
    for (score_key, raw_key, uid, offset, batch_n), out in zip(tasks, outputs):
        text = out.outputs[0].text
        scores = parse_scores_json(text, batch_n)
        if scores is None:
            scores = np.full(batch_n, (SCORE_MIN + SCORE_MAX) / 2.0)  # фолбэк: нейтральный
        full = pending.setdefault((score_key, uid), np.full(K_CANDIDATES, np.nan))
        full[offset:offset + batch_n] = scores
        raw_pending.setdefault((raw_key, uid), []).append(text)

    for (score_key, uid), arr in pending.items():
        if np.any(np.isnan(arr)):
            mean = float(np.nanmean(arr)) if not np.all(np.isnan(arr)) else (SCORE_MIN + SCORE_MAX) / 2.0
            arr = np.where(np.isnan(arr), mean, arr)
        pw_cache[score_key][uid] = arr

    for (raw_key, uid), texts in raw_pending.items():
        pw_cache[raw_key][uid] = '\n---\n'.join(texts)

with open(LLM_RERANK_PW_CACHE, 'w', encoding='utf-8') as f:
    json.dump(
        {k: ({str(uid): v.tolist() if isinstance(v, np.ndarray) else v
              for uid, v in d.items()})
         for k, d in pw_cache.items()},
        f, ensure_ascii=False
    )
print(f'Кэш сохранён в {LLM_RERANK_PW_CACHE}')


In [ ]:
for src_name, score_key in [('ALS', 'als_scores'), ('NCF', 'ncf_scores')]:
    all_scores = np.concatenate([
        v for v in pw_cache[score_key].values() if len(v) > 0
    ]) if pw_cache[score_key] else np.array([])
    if len(all_scores) == 0:
        print(f'{src_name}: нет скоров')
        continue
    counts = pd.Series(all_scores).value_counts(normalize=True).sort_index()
    mode_share = counts.max()
    print(f'{src_name}: всего {len(all_scores)} скоров, distribution:')
    for v, frac in counts.head(8).items():
        print(f'    {v:.2f}: {frac*100:5.1f}%')
    flag = ' ⚠️ MODE >90%' if mode_share > 0.9 else ''
    print(f'  Mode share: {mode_share*100:.1f}%{flag}')


In [ ]:
def rank_from_scores(cand_ids, scores_arr):
    """Возвращает dict: item_id -> rank (1..n) по убыванию score, tie-break по cand_ids порядку."""
    n = len(cand_ids)
    if scores_arr is None or len(scores_arr) != n:
        return {iid: i + 1 for i, iid in enumerate(cand_ids)}
    order = sorted(range(n), key=lambda i: (-scores_arr[i], i))
    return {cand_ids[order[k]]: k + 1 for k in range(n)}


def rrf_pointwise_recommend(uid: int, cand_dict, score_dict, alpha: float, k_rrf: int = RRF_K):
    cand_pairs = cand_dict[int(uid)]
    cand_ids = [iid for iid, _ in cand_pairs]
    n = len(cand_ids)
    if n == 0:
        return []
    rank_als = {iid: i + 1 for i, iid in enumerate(cand_ids)}
    scores_arr = score_dict.get(int(uid))
    rank_llm = rank_from_scores(cand_ids, scores_arr)
    final_scores = {
        iid: 1.0 / (k_rrf + rank_als[iid]) + alpha * 1.0 / (k_rrf + rank_llm[iid])
        for iid in cand_ids
    }
    return sorted(cand_ids, key=lambda x: -final_scores[x])[:FINAL_K]


def llm_only_pointwise_recommend(uid: int, cand_dict, score_dict):
    cand_pairs = cand_dict[int(uid)]
    cand_ids = [iid for iid, _ in cand_pairs]
    scores_arr = score_dict.get(int(uid))
    rank_llm = rank_from_scores(cand_ids, scores_arr)
    return sorted(cand_ids, key=lambda x: rank_llm[x])[:FINAL_K]


def build_recs_pointwise(cand_dict, score_dict, alpha):
    return {int(uid): rrf_pointwise_recommend(int(uid), cand_dict, score_dict, alpha)
            for uid in sample_users}


def build_recs_llm_only_pw(cand_dict, score_dict):
    return {int(uid): llm_only_pointwise_recommend(int(uid), cand_dict, score_dict)
            for uid in sample_users}


In [ ]:
K_VALUES = [5, 10, 20]


def precision_at_k(recommended, relevant, k):
    rec_k = set(recommended[:k])
    return len(rec_k & set(relevant)) / len(rec_k) if rec_k else 0.0


def recall_at_k(recommended, relevant, k):
    rec_k = set(recommended[:k])
    return len(rec_k & set(relevant)) / len(set(relevant)) if relevant else 0.0


def map_at_k(recommended, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    score, hits = 0.0, 0.0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(relevant), k)


def ndcg_at_k(recommended, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(recommended[:k]) if item in relevant)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0


def aggregate(recs_by_user, users):
    out = {k: {m: [] for m in ['precision', 'recall', 'map', 'ndcg']} for k in K_VALUES}
    for u in users:
        u = int(u)
        rel = test_user_items.get(u, [])
        if not rel:
            continue
        rec = recs_by_user.get(u, [])
        for k in K_VALUES:
            out[k]['precision'].append(precision_at_k(rec, rel, k))
            out[k]['recall'].append(recall_at_k(rec, rel, k))
            out[k]['map'].append(map_at_k(rec, rel, k))
            out[k]['ndcg'].append(ndcg_at_k(rec, rel, k))
    return {k: {m: float(np.mean(v)) if v else 0.0 for m, v in metrics.items()}
            for k, metrics in out.items()}


als_recs_base = {int(uid): [iid for iid, _ in als_candidates[int(uid)][:FINAL_K]] for uid in sample_users}
ncf_recs_base = {int(uid): [iid for iid, _ in ncf_candidates[int(uid)][:FINAL_K]] for uid in sample_users}

sanity_als = build_recs_pointwise(als_candidates, pw_cache['als_scores'], alpha=0.0)
match = all(sanity_als[u] == als_recs_base[u] for u in sample_users)
print(f'Sanity (pointwise α=0 vs base ALS): {"OK" if match else "FAIL"}')

sources_for_eval = [
    ('ALS', als_candidates, 'als_scores', als_recs_base),
    ('NCF', ncf_candidates, 'ncf_scores', ncf_recs_base),
]

all_results = {}
for src_name, cand_dict, score_key, base_recs in sources_for_eval:
    all_results[src_name] = aggregate(base_recs, sample_users)
    all_results[f'{src_name} + LLM only'] = aggregate(
        build_recs_llm_only_pw(cand_dict, pw_cache[score_key]), sample_users
    )
    for alpha in ALPHA_SWEEP:
        recs = build_recs_pointwise(cand_dict, pw_cache[score_key], alpha)
        all_results[f'{src_name} + LLM pw (α={alpha})'] = aggregate(recs, sample_users)

print(f'Конфигураций: {len(all_results)}')


In [ ]:
rows = []
for name, res in all_results.items():
    row = {'model': name}
    for k in K_VALUES:
        for m in ('precision', 'recall', 'map', 'ndcg'):
            row[f'{m}@{k}'] = res[k][m]
    rows.append(row)
comp_df = pd.DataFrame(rows).set_index('model')
print('Pointwise rerank — сводная таблица:')
display(comp_df.style.format('{:.4f}'))


In [ ]:

delta_rows = []
for src_name in ['ALS', 'NCF']:
    base_res = all_results[src_name]
    for label_suffix in ['LLM only'] + [f'LLM pw (α={a})' for a in ALPHA_SWEEP]:
        key = f'{src_name} + {label_suffix}'
        if key not in all_results:
            continue
        rer_res = all_results[key]
        delta = {
            f'{m}@{k}': (rer_res[k][m] - base_res[k][m]) / (base_res[k][m] + 1e-12) * 100
            for k in K_VALUES for m in ('precision', 'recall', 'map', 'ndcg')
        }
        delta_rows.append({'pair': f'{src_name} → {label_suffix}', **delta})

delta_df = pd.DataFrame(delta_rows).set_index('pair')
print('Δ относительно базы (%):')
display(delta_df.style.format('{:+.1f}%'))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
metrics_list = ['precision', 'recall', 'map', 'ndcg']
colors = {'ALS': 'tab:blue', 'NCF': 'tab:orange'}
linestyles = {0.0: ':', 0.1: '-.', 0.3: '--', 0.5: '-', 1.0: (0, (3, 1, 1, 1))}

for ax, metric in zip(axes.flat, metrics_list):
    for src_name in ['ALS', 'NCF']:
        base_res = all_results[src_name]
        ax.plot(K_VALUES, [base_res[k][metric] for k in K_VALUES],
                marker='o', linestyle='-', linewidth=2,
                color=colors[src_name], label=f'{src_name} (base)')
        llm_only_res = all_results.get(f'{src_name} + LLM only')
        if llm_only_res:
            ax.plot(K_VALUES, [llm_only_res[k][metric] for k in K_VALUES],
                    marker='D', linestyle=(0, (1, 1)), linewidth=1.5,
                    color=colors[src_name], alpha=0.9,
                    label=f'{src_name} LLM only')
        for alpha in ALPHA_SWEEP:
            label = f'{src_name} + LLM pw (α={alpha})'
            res = all_results[label]
            ax.plot(K_VALUES, [res[k][metric] for k in K_VALUES],
                    marker='s', linestyle=linestyles[alpha], linewidth=1,
                    color=colors[src_name], alpha=0.7, label=f'{src_name} α={alpha}')
    ax.set_xlabel('K')
    ax.set_ylabel(f'{metric.upper()}@K')
    ax.set_title(f'{metric.upper()}@K')
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)
plt.suptitle(f'Pointwise LLM rerank: alpha sweep (n={len(sample_users)})', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
uid = int(sample_users[0])
print(f'=== uid={uid} ===\n')
print(format_user_profile(uid))

print('\n--- ALS top-10 (base) ---')
for iid, sc in als_candidates[uid][:10]:
    feat = item_feat_dict.get(iid, {})
    print(f'  {iid}: {item_name_map.loc[iid][:55]} | {feat.get("Группа2", "?")} | {feat.get("item_avg_price", 0):.0f}р | als={sc:.3f}')

print('\n--- LLM pointwise scores для ALS top-10 ---')
scores = pw_cache['als_scores'].get(uid, np.array([]))
if len(scores):
    cand_ids = [iid for iid, _ in als_candidates[uid][:10]]
    for i, iid in enumerate(cand_ids):
        print(f'  {iid}: score={scores[i]:.2f}')

print('\n--- ALS top-10 после pointwise α=0.5 ---')
recs = rrf_pointwise_recommend(uid, als_candidates, pw_cache['als_scores'], alpha=0.5)
for iid in recs[:10]:
    feat = item_feat_dict.get(iid, {})
    print(f'  {iid}: {item_name_map.loc[iid][:55]} | {feat.get("Группа2", "?")}')

print('\n--- Test items ---')
for iid in test_user_items.get(uid, [])[:10]:
    print(f'  {iid}: {item_name_map.loc[iid][:55]}')


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd().resolve()
for _cand in [_here, *_here.parents]:
    _utils = _cand / 'source' / 'utils'
    if _utils.exists():
        sys.path.insert(0, str(_utils))
        break
import importlib, timing as _t
importlib.reload(_t)
from timing import benchmark_user_latency, save_benchmark, detect_hardware, benchmark_amortized_total

import math
rng = np.random.RandomState(42)
_users_arr = np.array(sample_users)
bench_size = min(2000, len(_users_arr))
bench_users = sorted(map(int, rng.choice(_users_arr, size=bench_size, replace=False).tolist()))
_batches_per_user = math.ceil(K_CANDIDATES / BATCH_SIZE)
print(f'Benchmark на {len(bench_users)} users (warmup=50, k=10, K_CANDIDATES={K_CANDIDATES}, BATCH_SIZE={BATCH_SIZE}, batches/user={_batches_per_user})')

import torch
DATASET_META = {'n_users': int(n_users), 'n_items': int(n_items),
                'n_candidates': int(K_CANDIDATES), 'final_k': int(FINAL_K),
                'pw_batch_size': int(BATCH_SIZE)}
NB_PATH = 'source/collaborative_filtering/llm_rerank_pointwise_v2.ipynb'

In [ ]:
try:
    _als_cpu = ALS_CPU
except NameError:
    from implicit.als import AlternatingLeastSquares as _ALS_CPU_CLS
    _als_cpu = _ALS_CPU_CLS(factors=50, regularization=0.01, iterations=15,
                            calculate_training_loss=False, random_state=RANDOM_SEED)
    _als_cpu.fit(train_matrix, show_progress=False)

def als_pw_stage1(uid):
    item_ids, scores = _als_cpu.recommend(
        uid, train_matrix[uid], N=K_CANDIDATES, filter_already_liked_items=True
    )
    return list(item_ids)

stats_als = benchmark_user_latency(
    als_pw_stage1, bench_users, warmup=50, k=10, sync_cuda=False, label='LLM-pw stage1 ALS-CPU'
)
print(f"ALS-CPU stage1: mean={stats_als['mean_ms']:.3f}ms p50={stats_als['p50_ms']:.3f} p95={stats_als['p95_ms']:.3f}")
save_benchmark(stats_als, model_name='ALS+LLM-pointwise', stage='stage1',
    hardware=detect_hardware(prefer='cpu'), dataset_meta=DATASET_META,
    extra={'library': 'implicit', 'use_gpu': False, 'stage_desc': 'ALS top-50'},
    notebook=NB_PATH, n_items=n_items)

In [ ]:
# --- stage1 NCF (GPU/CPU) ---
ncf_model.eval()
ncf_model.to(device)
import torch
_items_t_full = torch.arange(n_items, device=device, dtype=torch.long)

@torch.no_grad()
def ncf_pw_stage1(uid):
    u = torch.full_like(_items_t_full, fill_value=int(uid))
    scores = ncf_model(u, _items_t_full)
    bought = user_train_items_cache.get(int(uid), set())
    if bought:
        idx = [it for it in bought if it < n_items]
        if idx:
            scores[torch.tensor(idx, device=device, dtype=torch.long)] = float('-inf')
    top_idx = torch.topk(scores, K_CANDIDATES).indices.cpu().numpy()
    return list(top_idx)

stats_ncf = benchmark_user_latency(
    ncf_pw_stage1, bench_users, warmup=100, k=10, sync_cuda=(device.type == 'cuda'),
    label='LLM-pw stage1 NCF'
)
print(f"NCF stage1 ({device}): mean={stats_ncf['mean_ms']:.3f}ms p50={stats_ncf['p50_ms']:.3f} p95={stats_ncf['p95_ms']:.3f}")
save_benchmark(stats_ncf, model_name='NCF+LLM-pointwise', stage='stage1',
    hardware=detect_hardware(prefer='gpu') if device.type == 'cuda' else detect_hardware(prefer='cpu'),
    dataset_meta=DATASET_META,
    extra={'library': 'torch_neumf', 'sync_cuda': device.type == 'cuda',
           'stage_desc': 'NCF top-50'},
    notebook=NB_PATH, n_items=n_items)

In [ ]:
from vllm import SamplingParams
_sp = SamplingParams(temperature=0.0, max_tokens=300, stop_token_ids=LLM_TERMINATORS)

N_USERS_FOR_LLM = min(200, len(bench_users))
batch_users_for_llm = bench_users[:N_USERS_FOR_LLM]

def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

batch_prompts = []
for uid in batch_users_for_llm:
    cands = als_candidates.get(uid, [])
    if not cands:
        continue
    cand_ids = [iid for iid, _ in cands]
    for chunk in chunked(cand_ids, BATCH_SIZE):
        batch_prompts.append(build_pointwise_prompt(uid, chunk))
print(f'Промптов всего: {len(batch_prompts)}')

# Warmup
if len(batch_prompts) >= 2:
    _ = llm.generate(batch_prompts[:2], _sp)

import time
t0 = time.perf_counter()
_ = llm.generate(batch_prompts, _sp)
elapsed_batch = time.perf_counter() - t0

llm_per_prompt_ms = (elapsed_batch / len(batch_prompts)) * 1000.0
llm_per_user_ms = llm_per_prompt_ms * _batches_per_user
stats_llm_batch = benchmark_amortized_total(
    elapsed_batch * _batches_per_user, len(batch_prompts),  # total time per prompt-equivalent
    label='LLM-pointwise batch_amortized', k=10, batch_size=len(batch_prompts),
    note=f'amortized per-user = per-prompt × {_batches_per_user} batches/user'
)
stats_llm_batch['mean_ms'] = llm_per_user_ms
stats_llm_batch['n_users'] = N_USERS_FOR_LLM
stats_llm_batch['total_s'] = elapsed_batch
stats_llm_batch['throughput_ups'] = N_USERS_FOR_LLM / elapsed_batch if elapsed_batch > 0 else 0.0
print(f"LLM stage2 (pointwise): {len(batch_prompts)} prompts in {elapsed_batch:.1f}s → {llm_per_prompt_ms:.1f}ms/prompt, {llm_per_user_ms:.1f}ms/user")
save_benchmark(stats_llm_batch, model_name='LLM-pointwise-stage', stage='llm_rerank',
    hardware=detect_hardware(prefer='gpu'), dataset_meta=DATASET_META,
    extra={'library': 'vllm', 'mode': 'pointwise_batch_amortized',
           'continuous_batching': True, 'sync_cuda': True,
           'batches_per_user': _batches_per_user,
           'stage_desc': f'vLLM pointwise scoring: K={K_CANDIDATES}/B={BATCH_SIZE} per user'},
    notebook=NB_PATH, n_items=n_items)

In [ ]:
for base_name, stats_s1, model_name in [
    ('ALS-CPU', stats_als, 'ALS+LLM-pointwise'),
    ('NCF', stats_ncf, 'NCF+LLM-pointwise'),
]:
    e2e_lat = np.array(stats_s1['latencies_ms']) + llm_per_user_ms
    stats_e2e = {
        'mean_ms': float(e2e_lat.mean()),
        'p50_ms': float(np.percentile(e2e_lat, 50)),
        'p95_ms': float(np.percentile(e2e_lat, 95)),
        'total_s': float(e2e_lat.sum() / 1000.0),
        'throughput_ups': float(len(e2e_lat) * 1000.0 / e2e_lat.sum()),
        'label': f'{base_name}+LLM-pw e2e', 'k': 10,
        'n_users': len(e2e_lat), 'warmup': stats_s1['warmup'],
        'latencies_ms': e2e_lat.tolist(), 'mode': 'single_user',
        'note': 'stage1 measured + LLM pointwise amortized per-user',
    }
    print(f"{model_name} e2e: mean={stats_e2e['mean_ms']:.1f}ms p50={stats_e2e['p50_ms']:.1f} p95={stats_e2e['p95_ms']:.1f}")
    save_benchmark(stats_e2e, model_name=model_name, stage='e2e',
        hardware=detect_hardware(prefer='gpu'), dataset_meta=DATASET_META,
        extra={'base': base_name, 'composition': 'stage1 + llm_pointwise_amortized',
               'llm_per_user_ms': llm_per_user_ms},
        notebook=NB_PATH, n_items=n_items)